# PRISM-GAN — Notebook 04: Baselines (train, evaluate, compare) — Colab **L4 GPU**

## Baselines (no PRISM-GAN novelty in any of them)
| Run folder | Method | Why it is a baseline for this problem |
|---|---|---|
| `stylegan2_ada` | StyleGAN2 + Adaptive Discriminator Augmentation (Karras et al., NeurIPS 2020), class-conditional | the standard low-data GAN; used for textile patterns in the survey (African wax prints [14], ColorwAI [7], batik [12]) |
| `diffusion_gan` | Diffusion-GAN on the same StyleGAN2 backbone (Wang et al., ICLR 2023) | the batik-motif work in the survey [12] builds on it; a different stabiliser (instance-noise diffusion) on the same backbone |
| `biggan_diffaug` | BigGAN (Brock et al., ICLR 2019) + DiffAugment (Zhao et al., NeurIPS 2020) | canonical class-conditional GAN with the standard data-efficient augmentation; a different architecture family |

All three are compact re-implementations trained under **one protocol** (same data, class-conditioning, batch, epochs, evaluation). They are not the official code; the documented simplifications are listed at the top of `prism_baselines.py` and must be stated in the paper.

## Metrics (11) — defined once in `prism_metrics.py`, reused unchanged for PRISM-GAN and all ablations
FID ↓ · KID×10³ ↓ (**primary; selects the best checkpoint**) · FD-DINOv2 ↓ · Precision ↑ · Recall ↑ · Vendi ↑ · LPIPS diversity ↑ · Memorization rate ↓ · NIMA aesthetic ↑ · Seam ratio ↓ · Conditional accuracy ↑
Reference set = TRAIN split (3,781). The real TEST split (953) evaluated the same way gives the *real data* row of every table.
Per-epoch metrics use 2,000 fixed-latent samples; the final metrics of the best checkpoint use 10,000 samples.

## Protocol / storage
- Data: `patches_128/patches_128.zip` + `manifest_v2.csv` (verified by checksum), train images kept on the GPU; minibatches class-balanced with sqrt weighting.
- Every epoch: metrics printed, one row appended to `history.csv`, `checkpoints/last.pt` overwritten (atomic), `best.pt` updated when KID improves.
- **Resume:** just rerun the notebook. Each run continues from `last.pt`; `history.csv` is continued (the checkpoint stores its own copy of the history, so rows are never lost or duplicated). Finished runs are skipped.
- Outputs: `PRISM-GAN/baseline_outputs/<run>/` and `baseline_outputs/comparison/`. The comparison code accepts extra groups (`proposed_outputs`, `ablation_outputs`) so later notebooks regenerate the same tables/plots with PRISM-GAN included.
- Plots: font size 20, dpi 300.

**Budget:** `EPOCHS` is the training budget per baseline (1 epoch = 118 iterations of batch 32 = one pass over the train set). After epoch 1 the log prints an ETA; the same `EPOCHS` must later be used for PRISM-GAN and ablations.

## 0. Configuration

In [1]:
import os
ROOT = "/content/drive/MyDrive/hari/PRISM-GAN"
DATA = f"{ROOT}/datasets/DeepFashion-MultiModal/patches_128"
CODE = f"{ROOT}/code"                    # shared modules (metrics / runner / comparison) reused by later notebooks
CACHE = f"{ROOT}/metric_cache"           # real-data features + real-data reference metrics
OUT = f"{ROOT}/baseline_outputs"
LOCAL = "/content/prism_local"

BASELINES = ["stylegan2_ada", "diffusion_gan", "biggan_diffaug"]
EPOCHS = 200            # training budget per run (identical for PRISM-GAN and ablations)
EVAL_EVERY = 2          # metrics every epoch
N_EVAL = 2000           # samples per epoch-wise evaluation
N_FINAL = 10000         # samples for the final evaluation of best.pt
SAMPLE_EVERY = 10        # sample grid every N epochs
SEED = 42

CFG = dict(res=128, n_classes=4, batch=32, balance="sqrt",
           # StyleGAN2 backbone (B1, B2)
           z_dim=256, max_ch=256, lr=0.0025, r1_interval=16, r1_gamma=None,   # None -> ADA heuristic 0.0002*res^2/batch
           ada_kimg=500,
           # Diffusion-GAN (B2)
           dg_t_min=10, dg_t_max=1000, dg_beta_start=1e-4, dg_beta_end=2e-2, dg_sigma=0.05, dg_ada_kimg=100,
           # BigGAN + DiffAugment (B3)
           biggan_ch=32, biggan_lr_g=1e-4, biggan_lr_d=4e-4, biggan_n_critic=2, biggan_ema=0.999)
for d in [CODE, CACHE, OUT]: pass   # created after mounting Drive

## 1. Environment, Drive, GPU

In [2]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-fidelity==0.4.0", "pyiqa"], check=True)
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
for d in [CODE, CACHE, OUT, LOCAL, f"{LOCAL}/code"]: os.makedirs(d, exist_ok=True)
probe = f"{OUT}/.write_probe"
with open(probe, "w") as f: f.write("ok")
assert open(probe).read() == "ok"; os.remove(probe)

import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> L4 GPU"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
DEVICE = "cuda"

Mounted at /content/drive
GPU: NVIDIA L4 | torch 2.11.0+cu128


## 2. Shared code modules (written locally, copied to `PRISM-GAN/code/` for reuse)

In [3]:
%%writefile /content/prism_local/code/prism_metrics.py
"""PRISM-GAN shared evaluation module (used unchanged by baselines, proposed model and ablations).

Metrics (generated samples vs. the TRAIN split unless stated):
  fid           Frechet Inception Distance, Inception-v3 pool3 (torch-fidelity 'inception-v3-compat')      lower
  kid_x1e3      Kernel Inception Distance x 1e3 (poly kernel, 100 subsets of min(1000, N))                lower
  fd_dinov2     Frechet distance in DINOv2 ViT-B/14 feature space                                           lower
  precision     Improved precision, k=3, VGG16 fc2_relu (Kynkaanniemi et al. 2019, torch-fidelity)         higher
  recall        Improved recall, same setup                                                                 higher
  vendi         Vendi score (cosine kernel on DINOv2 features) - sample diversity                           higher
  lpips_div     Mean LPIPS(VGG16) between random pairs of generated samples - perceptual diversity          higher
  mem_rate      % of samples whose DINOv2 NN cosine distance to TRAIN < tau; tau = 5th percentile of the
                TEST->TRAIN NN distances (an unseen real image is this close only 5% of the time)            lower
  nima          NIMA aesthetic score (AVA-trained Inception-ResNet-v2, pyiqa 'nima'; pyiqa resizes the input), 1-10                                      higher
  seam_ratio    tileability: mean |wrap-around neighbour difference| / mean |interior neighbour difference|;
                1.0 = seams indistinguishable from interior                                                  lower (->1)
  cond_acc      accuracy of a logistic-regression probe (DINOv2 features, fit on TRAIN) on the class the
                sample was generated for - conditional fidelity                                              higher
"""
import json, math, numpy as np, torch, torch.nn.functional as F

METRICS = ["fid", "kid_x1e3", "fd_dinov2", "precision", "recall", "vendi",
           "lpips_div", "mem_rate", "nima", "seam_ratio", "cond_acc"]
DIRECTION = {"fid": "min", "kid_x1e3": "min", "fd_dinov2": "min", "precision": "max", "recall": "max",
             "vendi": "max", "lpips_div": "max", "mem_rate": "min", "nima": "max", "seam_ratio": "min",
             "cond_acc": "max"}
LABELS = {"fid": "FID", "kid_x1e3": "KID (x10$^3$)", "fd_dinov2": "FD$_{DINOv2}$", "precision": "Precision",
          "recall": "Recall", "vendi": "Vendi score", "lpips_div": "LPIPS diversity", "mem_rate": "Memorization (%)",
          "nima": "NIMA aesthetic", "seam_ratio": "Seam ratio", "cond_acc": "Conditional accuracy"}
PRIMARY = "kid_x1e3"


def to_uint8(x):
    """[-1,1] float NCHW -> uint8 NCHW."""
    return ((x.clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8)


def seam_ratio(u8):
    x = u8.float()
    interior = 0.5 * ((x[..., 1:, :] - x[..., :-1, :]).abs().mean((1, 2, 3)) +
                      (x[..., :, 1:] - x[..., :, :-1]).abs().mean((1, 2, 3)))
    seam = 0.5 * ((x[..., 0, :] - x[..., -1, :]).abs().mean((1, 2)) +
                  (x[..., :, 0] - x[..., :, -1]).abs().mean((1, 2)))
    return float((seam / interior.clamp_min(1e-6)).mean())


def vendi_score(feats):
    X = F.normalize(feats.double(), dim=1)
    n = X.shape[0]
    ev = torch.linalg.eigvalsh(X.T @ X / n).clamp_min(0)      # same non-zero spectrum as K/n
    ev = ev[ev > 1e-12]
    return float(torch.exp(-(ev * ev.log()).sum()))


def frechet(a, b):
    from torch_fidelity.metric_fid import fid_features_to_statistics, fid_statistics_to_metric, KEY_METRIC_FID
    return fid_statistics_to_metric(fid_features_to_statistics(a.double().cpu()),
                                    fid_features_to_statistics(b.double().cpu()), verbose=False)[KEY_METRIC_FID]


class MetricEngine:
    """Build once per session. Real TRAIN/TEST features are computed once and cached to disk."""

    def __init__(self, device, batch_size=100, extractors=None):
        self.device, self.bs = device, batch_size
        if extractors is None:
            from torch_fidelity.feature_extractor_inceptionv3 import FeatureExtractorInceptionV3
            from torch_fidelity.feature_extractor_vgg16 import FeatureExtractorVGG16
            from torch_fidelity.feature_extractor_dinov2 import FeatureExtractorDinoV2
            from torch_fidelity.sample_similarity_lpips import SampleSimilarityLPIPS
            import pyiqa
            extractors = dict(
                inception=FeatureExtractorInceptionV3("inception-v3-compat", ["2048"]),
                vgg=FeatureExtractorVGG16("vgg16", ["fc2_relu"]),
                dino=FeatureExtractorDinoV2("dinov2-vit-b-14", ["dinov2"]),
                lpips=SampleSimilarityLPIPS("lpips-vgg16", sample_similarity_dtype="uint8"),
                nima=pyiqa.create_metric("nima", device=device, as_loss=False),
            )
        self.ex = extractors
        for k in ("inception", "vgg", "dino", "lpips"):
            self.ex[k] = self.ex[k].to(device).eval()
        self.real = None

    @torch.no_grad()
    def _feats(self, u8):
        out = {"inception": [], "vgg": [], "dino": [], "nima": []}
        for i in range(0, len(u8), self.bs):
            b = u8[i:i + self.bs].to(self.device)
            out["inception"].append(self.ex["inception"](b)[0].float().cpu())
            out["vgg"].append(self.ex["vgg"](b)[0].float().cpu())
            out["dino"].append(self.ex["dino"](b)[0].float().cpu())
            out["nima"].append(self.ex["nima"](b.float() / 255.).reshape(-1).float().cpu())   # pyiqa resizes internally
        return {k: torch.cat(v) for k, v in out.items()}

    @torch.no_grad()
    def _lpips_div(self, u8, n_pairs=1000, seed=0):
        g = torch.Generator().manual_seed(seed)
        i = torch.randint(0, len(u8), (n_pairs,), generator=g)
        j = (i + torch.randint(1, len(u8), (n_pairs,), generator=g)) % len(u8)
        d = []
        for s in range(0, n_pairs, self.bs):
            a, b = u8[i[s:s + self.bs]].to(self.device), u8[j[s:s + self.bs]].to(self.device)
            d.append(self.ex["lpips"](a, b).reshape(-1).float().cpu())
        return float(torch.cat(d).mean())

    def build_real_cache(self, train_u8, train_y, test_u8, test_y, path=None, meta=None):
        from sklearn.linear_model import LogisticRegression
        tr, te = self._feats(train_u8), self._feats(test_u8)
        probe = LogisticRegression(max_iter=5000, C=1.0)
        probe.fit(F.normalize(tr["dino"], dim=1).numpy(), np.asarray(train_y))
        dn_tr = F.normalize(tr["dino"], dim=1)
        nn_te = 1 - (F.normalize(te["dino"], dim=1) @ dn_tr.T).max(1).values
        self.real = dict(train=tr, test=te, train_y=torch.as_tensor(train_y), test_y=torch.as_tensor(test_y),
                         probe_coef=torch.tensor(probe.coef_), probe_intercept=torch.tensor(probe.intercept_),
                         probe_classes=torch.tensor(probe.classes_), mem_tau=float(torch.quantile(nn_te, 0.05)),
                         meta=meta or {})
        if path: torch.save(self.real, path)
        return self.real

    def load_real_cache(self, path):
        self.real = torch.load(path, map_location="cpu", weights_only=False)
        return self.real

    def _probe_predict(self, dino):
        r = self.real
        logits = F.normalize(dino, dim=1).double() @ r["probe_coef"].T.double() + r["probe_intercept"].double()
        if logits.shape[1] == 1:   # binary case in sklearn
            return r["probe_classes"][(logits[:, 0] > 0).long()]
        return r["probe_classes"][logits.argmax(1)]

    @torch.no_grad()
    def evaluate(self, gen_u8, gen_y, seed=0):
        """gen_u8: uint8 NCHW generated images; gen_y: class index each was generated for."""
        from torch_fidelity.metric_kid import kid_features_to_metric, KEY_METRIC_KID_MEAN
        from torch_fidelity.metric_prc import prc_features_to_metric
        assert self.real is not None, "build_real_cache / load_real_cache first"
        g, r = self._feats(gen_u8), self.real["train"]
        n = min(1000, len(g["inception"]), len(r["inception"]))
        kid = kid_features_to_metric(g["inception"], r["inception"], kid_subsets=100, kid_subset_size=n,
                                     verbose=False, rng_seed=2020)[KEY_METRIC_KID_MEAN]
        prc = prc_features_to_metric(g["vgg"], r["vgg"], prc_neighborhood=3, verbose=False)
        dn_g, dn_tr = F.normalize(g["dino"], dim=1), F.normalize(r["dino"], dim=1)
        nn = 1 - (dn_g @ dn_tr.T).max(1).values
        pred = self._probe_predict(g["dino"])
        return {
            "fid": frechet(g["inception"], r["inception"]),
            "kid_x1e3": kid * 1e3,
            "fd_dinov2": frechet(g["dino"], r["dino"]),
            "precision": float(prc["precision"]),
            "recall": float(prc["recall"]),
            "vendi": vendi_score(g["dino"]),
            "lpips_div": self._lpips_div(gen_u8, seed=seed),
            "mem_rate": float((nn < self.real["mem_tau"]).float().mean() * 100),
            "nima": float(g["nima"].mean()),
            "seam_ratio": seam_ratio(gen_u8),
            "cond_acc": float((pred == torch.as_tensor(gen_y).cpu()).float().mean()),
        }

    def real_reference(self, test_u8):
        """Same metrics for the real TEST split (vs TRAIN): the 'real data' row of every comparison table."""
        return self.evaluate(test_u8, self.real["test_y"])

Writing /content/prism_local/code/prism_metrics.py


In [4]:
%%writefile /content/prism_local/code/prism_baselines.py
"""PRISM-GAN baselines: compact PyTorch re-implementations trained under one shared protocol.

B1 stylegan2_ada   StyleGAN2 generator/discriminator + Adaptive Discriminator Augmentation (Karras et al., 2020)
B2 diffusion_gan   Same StyleGAN2 backbone + Diffusion-GAN instance-noise diffusion with adaptive T (Wang et al., 2023)
B3 biggan_diffaug  BigGAN (Brock et al., 2019) + DiffAugment color,translation,cutout (Zhao et al., 2020)

These are faithful-in-mechanism re-implementations, NOT the official code. Documented simplifications:
  * StyleGAN2: bilinear resampling instead of FIR (upfirdn) filters; no path-length regularisation;
    no mixed precision. Mapping net = 2 layers, style mixing 0.9, lazy R1 (as in the ADA 'auto' config).
  * ADA: 'bgc' pipeline (xflip, rot90, int-translation, iso/aniso scale, rotation, frac-translation,
    brightness, contrast, luma-flip, hue, saturation) applied with one grid_sample (reflection padding).
  * Diffusion-GAN: timestep enters the projection discriminator through a sinusoidal embedding
    concatenated with the class one-hot.
  * BigGAN: channel multiplier 32, EMA from step 0, one self-attention block at 64x64.
"""
import math, numpy as np, torch, torch.nn as nn, torch.nn.functional as F

SQRT2 = math.sqrt(2)


# ----------------------------------------------------------------------------- equalised-lr layers
class EqLinear(nn.Module):
    def __init__(self, i, o, bias=True, lr_mul=1.0, bias_init=0.0, act=False):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(o, i) / lr_mul)
        self.bias = nn.Parameter(torch.full([o], float(bias_init))) if bias else None
        self.wg, self.bg, self.act = lr_mul / math.sqrt(i), lr_mul, act

    def forward(self, x):
        x = F.linear(x, self.weight * self.wg, None if self.bias is None else self.bias * self.bg)
        return F.leaky_relu(x, 0.2) * SQRT2 if self.act else x


class EqConv2d(nn.Module):
    def __init__(self, i, o, k, bias=True, act=False):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(o, i, k, k))
        self.bias = nn.Parameter(torch.zeros(o)) if bias else None
        self.wg, self.pad, self.act = 1 / math.sqrt(i * k * k), k // 2, act

    def forward(self, x):
        x = F.conv2d(x, self.weight * self.wg, self.bias, padding=self.pad)
        return F.leaky_relu(x, 0.2) * SQRT2 if self.act else x


# ----------------------------------------------------------------------------- StyleGAN2
def sg2_channels(res, max_ch=256, base=8192):
    return {r: min(base // r, max_ch) for r in [2 ** i for i in range(2, int(math.log2(res)) + 1)]}


class Mapping(nn.Module):
    def __init__(self, z_dim, c_dim, w_dim, layers=2, lr_mul=0.01):
        super().__init__()
        self.z_dim, self.c_dim = z_dim, c_dim
        self.embed = EqLinear(c_dim, w_dim) if c_dim else None
        dims = [(z_dim if z_dim else 0) + (w_dim if c_dim else 0)] + [w_dim] * layers
        self.net = nn.Sequential(*[EqLinear(dims[i], dims[i + 1], lr_mul=lr_mul, act=True) for i in range(layers)])

    @staticmethod
    def _norm(x): return x * (x.square().mean(1, keepdim=True) + 1e-8).rsqrt()

    def forward(self, z, c):
        parts = []
        if self.z_dim: parts.append(self._norm(z))
        if self.c_dim: parts.append(self._norm(self.embed(c)))
        return self.net(torch.cat(parts, 1))


class ModConv(nn.Module):
    def __init__(self, i, o, k, w_dim, demod=True, up=False):
        super().__init__()
        self.affine = EqLinear(w_dim, i, bias_init=1.0)
        self.weight = nn.Parameter(torch.randn(o, i, k, k))
        self.wg, self.pad, self.demod, self.up, self.o = 1 / math.sqrt(i * k * k), k // 2, demod, up, o

    def forward(self, x, w):
        N, C, H, W = x.shape
        s = self.affine(w)                                                # N x C
        wt = self.weight[None] * self.wg * s[:, None, :, None, None]      # N x O x C x k x k
        if self.demod:
            wt = wt * (wt.square().sum([2, 3, 4], keepdim=True) + 1e-8).rsqrt()
        if self.up:
            x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            H, W = H * 2, W * 2
        x = F.conv2d(x.reshape(1, N * C, H, W), wt.reshape(N * self.o, C, *wt.shape[3:]), padding=self.pad, groups=N)
        return x.reshape(N, self.o, H, W)


class SynthLayer(nn.Module):
    def __init__(self, i, o, w_dim, up=False):
        super().__init__()
        self.conv = ModConv(i, o, 3, w_dim, up=up)
        self.noise_strength = nn.Parameter(torch.zeros([]))
        self.bias = nn.Parameter(torch.zeros(o))

    def forward(self, x, w, noise=True):
        x = self.conv(x, w)
        if noise:
            x = x + torch.randn(x.shape[0], 1, x.shape[2], x.shape[3], device=x.device) * self.noise_strength
        return F.leaky_relu(x + self.bias[None, :, None, None], 0.2) * SQRT2


class ToRGB(nn.Module):
    def __init__(self, i, w_dim):
        super().__init__()
        self.conv = ModConv(i, 3, 1, w_dim, demod=False)
        self.bias = nn.Parameter(torch.zeros(3))

    def forward(self, x, w): return self.conv(x, w) + self.bias[None, :, None, None]


class SG2Generator(nn.Module):
    def __init__(self, res=128, z_dim=256, w_dim=256, c_dim=4, map_layers=2, max_ch=256):
        super().__init__()
        self.z_dim, self.c_dim, self.res = z_dim, c_dim, res
        ch = sg2_channels(res, max_ch)
        self.resolutions = sorted(ch)
        self.mapping = Mapping(z_dim, c_dim, w_dim, map_layers)
        self.const = nn.Parameter(torch.randn(ch[4], 4, 4))
        self.convs, self.torgbs = nn.ModuleList(), nn.ModuleList()
        for r in self.resolutions:
            if r == 4:
                self.convs.append(nn.ModuleList([SynthLayer(ch[4], ch[4], w_dim)]))
            else:
                self.convs.append(nn.ModuleList([SynthLayer(ch[r // 2], ch[r], w_dim, up=True),
                                                 SynthLayer(ch[r], ch[r], w_dim)]))
            self.torgbs.append(ToRGB(ch[r], w_dim))
        self.num_ws = 2 * len(self.resolutions)          # = sum(convs) + final toRGB, e.g. 12 at 128x128
        self.register_buffer("w_avg", torch.zeros(w_dim))

    def ws(self, z, c, mix_prob=0.0, z2=None):
        w = self.mapping(z, c)
        ws = w[:, None].repeat(1, self.num_ws, 1)
        if mix_prob > 0 and z2 is not None and np.random.rand() < mix_prob:
            cut = np.random.randint(1, self.num_ws)
            ws[:, cut:] = self.mapping(z2, c)[:, None]
        return ws

    def synthesis(self, ws, noise=True):
        x = self.const[None].repeat(ws.shape[0], 1, 1, 1)
        img, idx = None, 0
        for layers, torgb in zip(self.convs, self.torgbs):
            for layer in layers:
                x = layer(x, ws[:, idx], noise); idx += 1
            y = torgb(x, ws[:, idx])             # toRGB shares its w with the next block's first conv (as in SG2)
            img = y if img is None else F.interpolate(img, scale_factor=2, mode="bilinear", align_corners=False) + y
        return img

    def forward(self, z, c, mix_prob=0.0, z2=None, noise=True):
        return self.synthesis(self.ws(z, c, mix_prob, z2), noise)


class DBlock(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.conv0, self.conv1 = EqConv2d(i, i, 3, act=True), EqConv2d(i, o, 3, act=True)
        self.skip = EqConv2d(i, o, 1, bias=False)

    def forward(self, x):
        y = F.avg_pool2d(self.skip(x), 2)
        x = F.avg_pool2d(self.conv1(self.conv0(x)), 2)
        return (x + y) * math.sqrt(0.5)


class SG2Discriminator(nn.Module):
    """ResNet D with minibatch-stddev and projection conditioning; cond_dim = class one-hot (+ t-embedding)."""

    def __init__(self, res=128, cond_dim=4, cmap_dim=256, max_ch=256):
        super().__init__()
        ch = sg2_channels(res, max_ch)
        self.fromrgb = EqConv2d(3, ch[res], 1, act=True)
        self.blocks = nn.Sequential(*[DBlock(ch[r], ch[r // 2]) for r in sorted(ch, reverse=True) if r > 4])
        self.conv = EqConv2d(ch[4] + 1, ch[4], 3, act=True)
        self.fc = EqLinear(ch[4] * 16, ch[4], act=True)
        self.out = EqLinear(ch[4], cmap_dim)
        self.cmap = Mapping(0, cond_dim, cmap_dim, layers=2, lr_mul=1.0)
        self.cmap_dim = cmap_dim

    @staticmethod
    def mbstd(x, group=4):
        N, C, H, W = x.shape
        G = min(group, N)
        if N % G: G = 1
        y = x.reshape(G, -1, C, H, W)
        y = (y - y.mean(0)).square().mean(0).add(1e-8).sqrt().mean([1, 2, 3])
        y = y.reshape(-1, 1, 1, 1).repeat(G, 1, H, W)
        return torch.cat([x, y], 1)

    def forward(self, img, cond):
        x = self.blocks(self.fromrgb(img))
        x = self.conv(self.mbstd(x))
        x = self.out(self.fc(x.flatten(1)))
        return (x * self.cmap(None, cond)).sum(1, keepdim=True) / math.sqrt(self.cmap_dim)


# ----------------------------------------------------------------------------- ADA augmentation ('bgc')
class ADAugment(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("p", torch.zeros([]))

    @staticmethod
    def _rot(t):
        c, s, z, o = torch.cos(t), torch.sin(t), torch.zeros_like(t), torch.ones_like(t)
        return torch.stack([torch.stack([c, -s, z], -1), torch.stack([s, c, z], -1), torch.stack([z, z, o], -1)], -2)

    @staticmethod
    def _scale(sx, sy):
        z, o = torch.zeros_like(sx), torch.ones_like(sx)
        return torch.stack([torch.stack([sx, z, z], -1), torch.stack([z, sy, z], -1), torch.stack([z, z, o], -1)], -2)

    @staticmethod
    def _trans(tx, ty):
        z, o = torch.zeros_like(tx), torch.ones_like(tx)
        return torch.stack([torch.stack([o, z, tx], -1), torch.stack([z, o, ty], -1), torch.stack([z, z, o], -1)], -2)

    def forward(self, x):
        p = float(self.p)
        if p <= 0: return x
        N, dev = x.shape[0], x.device
        on = lambda q=p: (torch.rand(N, device=dev) < q).float()
        M = torch.eye(3, device=dev).repeat(N, 1, 1)
        # blit: x-flip, 90-degree rotations, integer translation
        flip = 1 - 2 * (on() * torch.randint(0, 2, (N,), device=dev).float())
        M = M @ self._scale(flip, torch.ones(N, device=dev))
        M = M @ self._rot(on() * torch.randint(0, 4, (N,), device=dev).float() * (math.pi / 2))
        t = (torch.rand(N, 2, device=dev) * 2 - 1) * 0.125 * 2 * on()[:, None]
        M = M @ self._trans(torch.round(t[:, 0] * x.shape[3] / 2) * 2 / x.shape[3],
                            torch.round(t[:, 1] * x.shape[2] / 2) * 2 / x.shape[2])
        # geom: isotropic scale, pre-rotation, anisotropic scale, post-rotation, fractional translation
        s = torch.exp2(torch.randn(N, device=dev) * 0.2 * on())
        M = M @ self._scale(s, s)
        p_rot = 1 - math.sqrt(1 - p)
        M = M @ self._rot((torch.rand(N, device=dev) * 2 - 1) * math.pi * on(p_rot))
        a = torch.exp2(torch.randn(N, device=dev) * 0.2 * on())
        M = M @ self._scale(a, 1 / a)
        M = M @ self._rot((torch.rand(N, device=dev) * 2 - 1) * math.pi * on(p_rot))
        ft = torch.randn(N, 2, device=dev) * 0.125 * 2 * on()[:, None]
        M = M @ self._trans(ft[:, 0], ft[:, 1])
        grid = F.affine_grid(M[:, :2, :], list(x.shape), align_corners=False)
        x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
        # color: brightness, contrast, luma flip, hue rotation, saturation  (on [-1,1] images, luma axis)
        C = torch.eye(4, device=dev).repeat(N, 1, 1)
        v = torch.tensor([1, 1, 1, 0], device=dev, dtype=x.dtype) / math.sqrt(3)
        b = torch.randn(N, device=dev) * 0.2 * on()
        Tb = torch.eye(4, device=dev).repeat(N, 1, 1); Tb[:, :3, 3] = b[:, None]
        C = Tb @ C
        c = torch.exp2(torch.randn(N, device=dev) * 0.5 * on())
        C = torch.diag_embed(torch.stack([c, c, c, torch.ones_like(c)], 1)) @ C
        vv = torch.outer(v, v)
        i_flip = torch.randint(0, 2, (N,), device=dev).float() * on()
        C = (torch.eye(4, device=dev) - 2 * vv * i_flip[:, None, None]) @ C
        th = (torch.rand(N, device=dev) * 2 - 1) * math.pi * on()
        ax = v[:3]
        K = torch.tensor([[0, -ax[2], ax[1]], [ax[2], 0, -ax[0]], [-ax[1], ax[0], 0]], device=dev, dtype=x.dtype)
        R = torch.eye(3, device=dev) + torch.sin(th)[:, None, None] * K + (1 - torch.cos(th))[:, None, None] * (K @ K)
        Rh = torch.eye(4, device=dev).repeat(N, 1, 1); Rh[:, :3, :3] = R
        C = Rh @ C
        sat = torch.exp2(torch.randn(N, device=dev) * 1.0 * on())
        C = (vv + (torch.eye(4, device=dev) - vv) * sat[:, None, None]) @ C
        y = x.reshape(N, 3, -1)
        y = C[:, :3, :3] @ y + C[:, :3, 3:]
        return y.reshape_as(x)


# ----------------------------------------------------------------------------- DiffAugment (Zhao et al. 2020)
def diff_augment(x, policy="color,translation,cutout"):
    for p in policy.split(","):
        if p == "color":
            x = x + (torch.rand(x.size(0), 1, 1, 1, device=x.device) - 0.5)
            m = x.mean(dim=1, keepdim=True)
            x = (x - m) * (torch.rand(x.size(0), 1, 1, 1, device=x.device) * 2) + m
            m = x.mean(dim=[1, 2, 3], keepdim=True)
            x = (x - m) * (torch.rand(x.size(0), 1, 1, 1, device=x.device) + 0.5) + m
        elif p == "translation":
            sx, sy = int(x.size(2) * 0.125 + 0.5), int(x.size(3) * 0.125 + 0.5)
            tx = torch.randint(-sx, sx + 1, size=[x.size(0), 1, 1], device=x.device)
            ty = torch.randint(-sy, sy + 1, size=[x.size(0), 1, 1], device=x.device)
            gb, gx, gy = torch.meshgrid(torch.arange(x.size(0), device=x.device),
                                        torch.arange(x.size(2), device=x.device),
                                        torch.arange(x.size(3), device=x.device), indexing="ij")
            gx = torch.clamp(gx + tx + 1, 0, x.size(2) + 1)
            gy = torch.clamp(gy + ty + 1, 0, x.size(3) + 1)
            xp = F.pad(x, [1, 1, 1, 1, 0, 0, 0, 0])
            x = xp.permute(0, 2, 3, 1).contiguous()[gb, gx, gy].permute(0, 3, 1, 2)
        elif p == "cutout":
            cs = int(x.size(2) * 0.5 + 0.5), int(x.size(3) * 0.5 + 0.5)
            ox = torch.randint(0, x.size(2) + (1 - cs[0] % 2), size=[x.size(0), 1, 1], device=x.device)
            oy = torch.randint(0, x.size(3) + (1 - cs[1] % 2), size=[x.size(0), 1, 1], device=x.device)
            gb, gx, gy = torch.meshgrid(torch.arange(x.size(0), device=x.device),
                                        torch.arange(cs[0], device=x.device),
                                        torch.arange(cs[1], device=x.device), indexing="ij")
            gx = torch.clamp(gx + ox - cs[0] // 2, min=0, max=x.size(2) - 1)
            gy = torch.clamp(gy + oy - cs[1] // 2, min=0, max=x.size(3) - 1)
            mask = torch.ones(x.size(0), x.size(2), x.size(3), dtype=x.dtype, device=x.device)
            mask[gb, gx, gy] = 0
            x = x * mask.unsqueeze(1)
    return x


# ----------------------------------------------------------------------------- BigGAN
SN = nn.utils.spectral_norm


class CBN(nn.Module):
    def __init__(self, ch, cond_dim):
        super().__init__()
        self.bn = nn.BatchNorm2d(ch, affine=False)
        self.gain, self.bias = SN(nn.Linear(cond_dim, ch, bias=False)), SN(nn.Linear(cond_dim, ch, bias=False))

    def forward(self, x, y):
        return self.bn(x) * (1 + self.gain(y))[:, :, None, None] + self.bias(y)[:, :, None, None]


class GBlock(nn.Module):
    def __init__(self, i, o, cond_dim):
        super().__init__()
        self.bn1, self.bn2 = CBN(i, cond_dim), CBN(o, cond_dim)
        self.c1, self.c2 = SN(nn.Conv2d(i, o, 3, padding=1)), SN(nn.Conv2d(o, o, 3, padding=1))
        self.sc = SN(nn.Conv2d(i, o, 1))

    def forward(self, x, y):
        h = F.interpolate(F.relu(self.bn1(x, y)), scale_factor=2)
        h = self.c2(F.relu(self.bn2(self.c1(h), y)))
        return h + self.sc(F.interpolate(x, scale_factor=2))


class Attention(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.theta, self.phi = SN(nn.Conv2d(ch, ch // 8, 1, bias=False)), SN(nn.Conv2d(ch, ch // 8, 1, bias=False))
        self.g, self.o = SN(nn.Conv2d(ch, ch // 2, 1, bias=False)), SN(nn.Conv2d(ch // 2, ch, 1, bias=False))
        self.gamma = nn.Parameter(torch.zeros([]))

    def forward(self, x):
        N, C, H, W = x.shape
        th = self.theta(x).view(N, C // 8, H * W)
        ph = F.max_pool2d(self.phi(x), 2).view(N, C // 8, H * W // 4)
        g = F.max_pool2d(self.g(x), 2).view(N, C // 2, H * W // 4)
        beta = F.softmax(th.transpose(1, 2) @ ph, -1)
        o = self.o((g @ beta.transpose(1, 2)).view(N, C // 2, H, W))
        return x + self.gamma * o


class BigGANGenerator(nn.Module):
    def __init__(self, res=128, z_dim=120, n_classes=4, ch=32, shared_dim=128, attn_res=64):
        super().__init__()
        mult = {128: [16, 16, 8, 4, 2, 1], 64: [16, 16, 8, 4, 2], 32: [4, 4, 4, 4]}[res]
        self.n_blocks = len(mult) - 1
        self.z_chunk = z_dim // (self.n_blocks + 1)
        self.z_dim = self.z_chunk * (self.n_blocks + 1)
        self.c_dim = n_classes
        self.shared = nn.Embedding(n_classes, shared_dim)
        self.linear = SN(nn.Linear(self.z_chunk, ch * mult[0] * 16))
        self.blocks, r = nn.ModuleList(), 4
        self.attn_at = None
        for b in range(self.n_blocks):
            self.blocks.append(GBlock(ch * mult[b], ch * mult[b + 1], shared_dim + self.z_chunk))
            r *= 2
            if r == attn_res: self.attn_at, self.attn = b, Attention(ch * mult[b + 1])
        self.out_bn = nn.BatchNorm2d(ch * mult[-1])
        self.out = SN(nn.Conv2d(ch * mult[-1], 3, 3, padding=1))
        self.ch0 = ch * mult[0]

    def forward(self, z, c_idx):
        zs = torch.split(z, self.z_chunk, 1)
        e = self.shared(c_idx)
        h = self.linear(zs[0]).view(z.shape[0], self.ch0, 4, 4)
        for b, blk in enumerate(self.blocks):
            h = blk(h, torch.cat([e, zs[b + 1]], 1))
            if b == self.attn_at: h = self.attn(h)
        return torch.tanh(self.out(F.relu(self.out_bn(h))))


class DResBlock(nn.Module):
    def __init__(self, i, o, down=True, preact=True):
        super().__init__()
        self.c1, self.c2 = SN(nn.Conv2d(i, o, 3, padding=1)), SN(nn.Conv2d(o, o, 3, padding=1))
        self.sc = SN(nn.Conv2d(i, o, 1)) if (i != o or down) else None
        self.down, self.preact = down, preact

    def forward(self, x):
        h = F.relu(x) if self.preact else x
        h = self.c2(F.relu(self.c1(h)))
        if self.down: h = F.avg_pool2d(h, 2)
        s = x
        if self.preact:
            if self.sc is not None: s = self.sc(s)
            if self.down: s = F.avg_pool2d(s, 2)
        else:
            if self.down: s = F.avg_pool2d(s, 2)
            if self.sc is not None: s = self.sc(s)
        return h + s


class BigGANDiscriminator(nn.Module):
    def __init__(self, res=128, n_classes=4, ch=32, attn_res=64):
        super().__init__()
        mult = {128: [1, 2, 4, 8, 16, 16], 64: [1, 2, 4, 8, 16], 32: [4, 4, 4, 4]}[res]
        chs = [3] + [ch * m for m in mult]
        self.blocks, r = nn.ModuleList(), res
        for b in range(len(mult)):
            last = b == len(mult) - 1
            self.blocks.append(DResBlock(chs[b], chs[b + 1], down=not last, preact=b > 0))
            if not last: r //= 2
            if r == attn_res and b == 0: self.blocks.append(Attention(chs[b + 1]))
        self.linear = SN(nn.Linear(chs[-1], 1))
        self.embed = SN(nn.Embedding(n_classes, chs[-1]))

    def forward(self, x, c_idx):
        for blk in self.blocks: x = blk(x)
        h = F.relu(x).sum([2, 3])
        return self.linear(h) + (self.embed(c_idx) * h).sum(1, keepdim=True)


# ----------------------------------------------------------------------------- helpers
def ema_update(ema, model, beta):
    with torch.no_grad():
        for pe, p in zip(ema.parameters(), model.parameters()): pe.copy_(p.lerp(pe, beta))
        for be, b in zip(ema.buffers(), model.buffers()): be.copy_(b)


def t_embed(t, dim=16, t_max=1000):
    half = dim // 2
    f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    a = (t.float() / t_max * 1000)[:, None] * f[None]
    return torch.cat([a.sin(), a.cos()], 1)


def n_params(m): return sum(p.numel() for p in m.parameters())


# ----------------------------------------------------------------------------- trainers
class StyleGAN2Trainer:
    """B1 (augment='ada') and B2 (augment='diffusion') share this trainer."""

    def __init__(self, cfg, device, mode="ada"):
        import copy
        self.cfg, self.dev, self.mode = cfg, device, mode
        nc, res, bs = cfg["n_classes"], cfg["res"], cfg["batch"]
        self.T_EMB = 16
        cond = nc + (self.T_EMB if mode == "diffusion" else 0)
        self.G = SG2Generator(res, cfg["z_dim"], cfg["z_dim"], nc, 2, cfg["max_ch"]).to(device)
        self.D = SG2Discriminator(res, cond, 256, cfg["max_ch"]).to(device)
        self.G_ema = copy.deepcopy(self.G).eval().requires_grad_(False)
        lr, reg = cfg["lr"], cfg["r1_interval"]
        c = reg / (reg + 1)
        self.optG = torch.optim.Adam(self.G.parameters(), lr=lr, betas=(0.0, 0.99), eps=1e-8)
        self.optD = torch.optim.Adam(self.D.parameters(), lr=lr * c, betas=(0.0, 0.99 ** c), eps=1e-8)
        self.gamma = cfg.get("r1_gamma") or 0.0002 * res ** 2 / bs          # ADA heuristic
        self.aug = ADAugment().to(device)
        # Diffusion-GAN state
        self.t_min, self.t_max = cfg.get("dg_t_min", 10), cfg.get("dg_t_max", 1000)
        self.betas = torch.linspace(cfg.get("dg_beta_start", 1e-4), cfg.get("dg_beta_end", 2e-2), self.t_max, device=device)
        self.abar = torch.cumprod(1 - self.betas, 0)
        self.p_T = 0.0
        self.rt_acc, self.nimg, self.it = [], 0, 0

    # --- conditioning / augmentation
    def onehot(self, y): return F.one_hot(y, self.cfg["n_classes"]).float()

    def _T(self): return int(round(self.t_min + self.p_T * (self.t_max - self.t_min)))

    def _prep(self, x, y, t=None):
        c = self.onehot(y)
        if self.mode == "ada":
            return self.aug(x), c
        T = self._T()
        if t is None:
            w = torch.arange(1, T + 1, device=x.device, dtype=torch.float)            # 'priority' t sampling
            t = torch.multinomial(w / w.sum(), x.shape[0], replacement=True)
        ab = self.abar[t][:, None, None, None]
        x = ab.sqrt() * x + (1 - ab).sqrt() * self.cfg.get("dg_sigma", 0.05) * torch.randn_like(x)
        return x, torch.cat([c, t_embed(t, self.T_EMB, self.t_max)], 1), t

    def D_in(self, x, y, t=None):
        out = self._prep(x, y, t)
        return (out[0], out[1]) if self.mode == "ada" else (out[0], out[1], out[2])

    def gen(self, n, y, mix=True):
        z = torch.randn(n, self.cfg["z_dim"], device=self.dev)
        z2 = torch.randn(n, self.cfg["z_dim"], device=self.dev)
        return self.G(z, self.onehot(y), mix_prob=0.9 if mix else 0.0, z2=z2)

    # --- one training iteration
    def step(self, real, y_real, y_fake):
        cfg, bs = self.cfg, real.shape[0]
        logs = {}
        # G
        self.G.requires_grad_(True); self.D.requires_grad_(False)
        fake = self.gen(bs, y_fake)
        di = self.D_in(fake, y_fake)
        lg = F.softplus(-self.D(di[0], di[1])).mean()
        self.optG.zero_grad(set_to_none=True); lg.backward(); self.optG.step()
        # D
        self.G.requires_grad_(False); self.D.requires_grad_(True)
        with torch.no_grad(): fake = self.gen(bs, y_fake)
        df = self.D_in(fake, y_fake); dr = self.D_in(real, y_real)
        lf, lr_ = self.D(df[0], df[1]), self.D(dr[0], dr[1])
        ld = F.softplus(lf).mean() + F.softplus(-lr_).mean()
        self.optD.zero_grad(set_to_none=True); ld.backward(); self.optD.step()
        self.rt_acc.append(torch.sign(lr_.detach()).mean().item())
        # lazy R1 (on augmented / diffused reals, as in ADA)
        if self.it % cfg["r1_interval"] == 0:
            dr = self.D_in(real, y_real)
            xr = dr[0].detach().requires_grad_(True)
            out = self.D(xr, dr[1])
            g, = torch.autograd.grad(out.sum(), xr, create_graph=True)
            r1 = g.square().sum([1, 2, 3]).mean() * (self.gamma / 2) * cfg["r1_interval"]
            self.optD.zero_grad(set_to_none=True); r1.backward(); self.optD.step()
            logs["r1"] = r1.item() / cfg["r1_interval"]
        # adaptive p (ADA) / adaptive T (Diffusion-GAN): every 4 iterations, target r_t = 0.6
        if self.it % 4 == 0 and self.rt_acc:
            rt = float(np.mean(self.rt_acc)); self.rt_acc = []
            adj = np.sign(rt - 0.6) * (bs * 4) / ((cfg["ada_kimg"] if self.mode == "ada" else cfg["dg_ada_kimg"]) * 1000)
            if self.mode == "ada": self.aug.p.copy_((self.aug.p + adj).clamp(0, 1))
            else: self.p_T = float(np.clip(self.p_T + adj, 0, 1))
            logs["rt"] = rt
        # G EMA (ADA: ema_kimg = batch*10/32, ramp 0.05)
        self.nimg += bs
        ema_nimg = min(bs * 10 / 32 * 1000, self.nimg * 0.05)
        ema_update(self.G_ema, self.G, 0.5 ** (bs / max(ema_nimg, 1e-8)))
        self.it += 1
        logs.update(loss_G=lg.item(), loss_D=ld.item(),
                    aug_p=float(self.aug.p) if self.mode == "ada" else float("nan"),
                    diff_T=float(self._T()) if self.mode == "diffusion" else float("nan"))
        return logs

    @torch.no_grad()
    def sample(self, z, y):
        return self.G_ema(z, self.onehot(y), noise=True)

    def z_dim(self): return self.cfg["z_dim"]

    def state_dict(self):
        return dict(G=self.G.state_dict(), D=self.D.state_dict(), G_ema=self.G_ema.state_dict(),
                    optG=self.optG.state_dict(), optD=self.optD.state_dict(), aug_p=float(self.aug.p),
                    p_T=self.p_T, nimg=self.nimg, it=self.it, rt_acc=self.rt_acc)

    def load_state_dict(self, s):
        self.G.load_state_dict(s["G"]); self.D.load_state_dict(s["D"]); self.G_ema.load_state_dict(s["G_ema"])
        self.optG.load_state_dict(s["optG"]); self.optD.load_state_dict(s["optD"])
        self.aug.p.fill_(s["aug_p"]); self.p_T, self.nimg, self.it, self.rt_acc = s["p_T"], s["nimg"], s["it"], s["rt_acc"]

    def params(self): return dict(G=n_params(self.G), D=n_params(self.D))


class BigGANTrainer:
    def __init__(self, cfg, device):
        import copy
        self.cfg, self.dev = cfg, device
        nc, res = cfg["n_classes"], cfg["res"]
        self.G = BigGANGenerator(res, 120, nc, cfg["biggan_ch"]).to(device)
        self.D = BigGANDiscriminator(res, nc, cfg["biggan_ch"]).to(device)
        self.G_ema = copy.deepcopy(self.G).eval().requires_grad_(False)
        self.optG = torch.optim.Adam(self.G.parameters(), lr=cfg["biggan_lr_g"], betas=(0.0, 0.999), eps=1e-6)
        self.optD = torch.optim.Adam(self.D.parameters(), lr=cfg["biggan_lr_d"], betas=(0.0, 0.999), eps=1e-6)
        self.it, self.nimg = 0, 0

    def step(self, real, y_real, y_fake):
        bs, logs, aug = real.shape[0], {}, lambda x: diff_augment(x, "color,translation,cutout")
        # D (hinge)
        self.G.requires_grad_(False); self.D.requires_grad_(True)
        with torch.no_grad():
            fake = self.G(torch.randn(bs, self.G.z_dim, device=self.dev), y_fake)
        ld = F.relu(1 - self.D(aug(real), y_real)).mean() + F.relu(1 + self.D(aug(fake), y_fake)).mean()
        self.optD.zero_grad(set_to_none=True); ld.backward(); self.optD.step()
        logs["loss_D"] = ld.item()
        # G every n_critic D steps
        if self.it % self.cfg["biggan_n_critic"] == 0:
            self.G.requires_grad_(True); self.D.requires_grad_(False)
            fake = self.G(torch.randn(bs, self.G.z_dim, device=self.dev), y_fake)
            lg = -self.D(aug(fake), y_fake).mean()
            self.optG.zero_grad(set_to_none=True); lg.backward(); self.optG.step()
            ema_update(self.G_ema, self.G, self.cfg["biggan_ema"])
            logs["loss_G"] = lg.item()
        self.it += 1; self.nimg += bs
        logs.update(aug_p=float("nan"), diff_T=float("nan"))
        return logs

    @torch.no_grad()
    def sample(self, z, y):
        return self.G_ema(z[:, :self.G.z_dim], y)

    def z_dim(self): return self.G.z_dim

    def state_dict(self):
        return dict(G=self.G.state_dict(), D=self.D.state_dict(), G_ema=self.G_ema.state_dict(),
                    optG=self.optG.state_dict(), optD=self.optD.state_dict(), it=self.it, nimg=self.nimg)

    def load_state_dict(self, s):
        self.G.load_state_dict(s["G"]); self.D.load_state_dict(s["D"]); self.G_ema.load_state_dict(s["G_ema"])
        self.optG.load_state_dict(s["optG"]); self.optD.load_state_dict(s["optD"]); self.it, self.nimg = s["it"], s["nimg"]

    def params(self): return dict(G=n_params(self.G), D=n_params(self.D))


def make_trainer(name, cfg, device):
    if name == "stylegan2_ada": return StyleGAN2Trainer(cfg, device, "ada")
    if name == "diffusion_gan": return StyleGAN2Trainer(cfg, device, "diffusion")
    if name == "biggan_diffaug": return BigGANTrainer(cfg, device)
    raise ValueError(name)

Writing /content/prism_local/code/prism_baselines.py


In [5]:
%%writefile /content/prism_local/code/prism_runner.py
"""Shared run protocol for baselines, the proposed model and ablations.

Folder layout per run (out_root/<run_name>/):
  config.json            full run configuration
  history.csv            one row per epoch (losses, schedules, all metrics); resumed runs append, never overwrite
  checkpoints/last.pt    full training state (models, optimisers, schedules, RNG, best-so-far) - saved every epoch
  checkpoints/best.pt    G_ema weights at the epoch with the best PRIMARY metric (kid_x1e3)
  samples/epoch_XXXX.png fixed-latent sample grid (rows = classes)
  plots/                 per-metric epoch curves (font 20, dpi 300)
  metrics.json           final metrics of best.pt on N_FINAL samples (+ best epoch, params, timing)
  status.json            'training' / 'done'
"""
import os, json, time, math, random, shutil
import numpy as np, pandas as pd, torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from prism_metrics import METRICS, DIRECTION, LABELS, PRIMARY, to_uint8

FONT, DPI = 20, 300


def atomic_torch_save(obj, dst, tmp_dir="/tmp"):
    tmp = os.path.join(tmp_dir, os.path.basename(dst) + ".tmp")
    torch.save(obj, tmp)
    atomic_copy(tmp, dst); os.remove(tmp)


def atomic_copy(src, dst):
    part = dst + ".partial"
    shutil.copyfile(src, part)
    assert os.path.getsize(part) == os.path.getsize(src), f"size mismatch writing {dst}"
    os.replace(part, dst)


def atomic_text(text, dst, tmp_dir="/tmp"):
    tmp = os.path.join(tmp_dir, os.path.basename(dst) + ".tmp")
    with open(tmp, "w") as f: f.write(text)
    atomic_copy(tmp, dst); os.remove(tmp)


def save_grid(u8, nrow, path):
    n, c, h, w = u8.shape
    rows = math.ceil(n / nrow)
    canvas = np.full((rows * (h + 2), nrow * (w + 2), 3), 255, np.uint8)
    for i in range(n):
        r, q = divmod(i, nrow)
        canvas[r * (h + 2):r * (h + 2) + h, q * (w + 2):q * (w + 2) + w] = u8[i].permute(1, 2, 0).cpu().numpy()
    from PIL import Image
    tmp = os.path.join("/tmp", os.path.basename(path))
    Image.fromarray(canvas).save(tmp); atomic_copy(tmp, path); os.remove(tmp)


def rng_state():
    return dict(torch=torch.get_rng_state(), cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
                numpy=np.random.get_state(), python=random.getstate())


def set_rng_state(s):
    torch.set_rng_state(s["torch"].cpu().to(torch.uint8))     # RNG states must be CPU ByteTensors
    if s["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([c.cpu().to(torch.uint8) for c in s["cuda"]])
    np.random.set_state(s["numpy"]); random.setstate(s["python"])


class TrainData:
    """Train images live on the GPU as uint8; sampling weights give class-balanced ('sqrt') minibatches."""

    def __init__(self, x_u8, y, device, balance="sqrt"):
        self.x, self.y = x_u8.to(device), torch.as_tensor(y).long().to(device)
        counts = torch.bincount(self.y).float()
        w_class = {"none": torch.ones_like(counts), "sqrt": counts.rsqrt(), "full": 1 / counts}[balance]
        self.w = w_class[self.y]
        self.class_prob = (w_class * counts) / (w_class * counts).sum()     # label prior seen by D for reals
        self.n, self.device = len(self.y), device

    def epoch_batches(self, batch, epoch, seed):
        g = torch.Generator(device="cpu").manual_seed(seed * 100003 + epoch)
        n_it = self.n // batch
        idx = torch.multinomial(self.w.cpu(), n_it * batch, replacement=True, generator=g).view(n_it, batch)
        yf = torch.multinomial(self.class_prob.cpu(), n_it * batch, replacement=True, generator=g).view(n_it, batch)
        for i in range(n_it):
            ii = idx[i].to(self.device)
            yield self.x[ii].float() / 127.5 - 1, self.y[ii], yf[i].to(self.device)


@torch.no_grad()
def generate(trainer, z, y, bs=100):
    out = [to_uint8(trainer.sample(z[i:i + bs], y[i:i + bs])).cpu() for i in range(0, len(z), bs)]
    return torch.cat(out)


def eval_latents(z_dim, n, class_prior, seed, device):
    g = torch.Generator(device="cpu").manual_seed(seed)
    z = torch.randn(n, z_dim, generator=g)
    y = torch.multinomial(class_prior.cpu(), n, replacement=True, generator=g)
    return z.to(device), y.to(device)


def plot_history(hist, plot_dir, title_prefix=""):
    os.makedirs(plot_dir, exist_ok=True)
    plt.rcParams.update({"font.size": FONT})
    cols = [m for m in METRICS if m in hist] + [c for c in ["loss_G", "loss_D", "aug_p", "diff_T", "r1", "rt"]
                                                 if c in hist and hist[c].notna().any()]
    for m in cols:
        h = hist[["epoch", m]].dropna()
        if h.empty: continue
        fig, ax = plt.subplots(figsize=(10, 7))
        ax.plot(h.epoch, h[m], lw=2.5)
        ax.set_xlabel("Epoch"); ax.set_ylabel(LABELS.get(m, m)); ax.set_title(f"{title_prefix}{LABELS.get(m, m)}")
        ax.grid(alpha=.3); fig.tight_layout()
        tmp = f"/tmp/{m}.png"; fig.savefig(tmp, dpi=DPI); plt.close(fig)
        atomic_copy(tmp, os.path.join(plot_dir, f"{m}.png")); os.remove(tmp)


def run(name, make_trainer, cfg, data, engine, out_root, epochs, eval_every=1, n_eval=2000,
        sample_every=5, seed=42, log=print):
    out = os.path.join(out_root, name)
    for d in ["checkpoints", "samples", "plots"]: os.makedirs(os.path.join(out, d), exist_ok=True)
    atomic_text(json.dumps(cfg, indent=2, default=str), os.path.join(out, "config.json"))
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    trainer = make_trainer()
    hist, start, best = [], 1, {"value": None, "epoch": None}
    last = os.path.join(out, "checkpoints", "last.pt")
    hpath = os.path.join(out, "history.csv")
    if os.path.exists(last):
        ck = torch.load(last, map_location="cpu", weights_only=False)
        trainer.load_state_dict(ck["trainer"]); set_rng_state(ck["rng"])
        start, best = ck["epoch"] + 1, ck["best"]
        # the checkpoint carries its own copy of the history (authoritative even if a Drive write of history.csv was lost)
        hist = [r for r in ck.get("history", []) if r["epoch"] <= ck["epoch"]]
        if os.path.exists(hpath):
            h = pd.read_csv(hpath)
            dropped = int((h.epoch > ck["epoch"]).sum())
            if not hist: hist = h[h.epoch <= ck["epoch"]].to_dict("records")
            if dropped: log(f"[{name}] history.csv had {dropped} row(s) after the checkpoint epoch; they will be recomputed")
        atomic_text(pd.DataFrame(hist).to_csv(index=False), hpath)   # continue from exactly the checkpoint epoch
        log(f"[{name}] resumed from epoch {ck['epoch']} (best {PRIMARY}={best['value']} @ {best['epoch']})")
    if start > epochs:
        log(f"[{name}] already trained for {epochs} epochs"); return out
    atomic_text(json.dumps({"state": "training", "epochs_target": epochs}), os.path.join(out, "status.json"))
    z_eval, y_eval = eval_latents(trainer.z_dim(), n_eval, data.class_prob, seed + 1, data.device)
    nc = int(data.y.max()) + 1
    z_grid = eval_latents(trainer.z_dim(), 8 * nc, data.class_prob, seed + 2, data.device)[0]
    y_grid = torch.arange(nc, device=data.device).repeat_interleave(8)
    t_run = time.time()
    for epoch in range(start, epochs + 1):
        t0, acc = time.time(), {}
        for real, yr, yf in data.epoch_batches(cfg["batch"], epoch, seed):
            for k, v in trainer.step(real, yr, yf).items(): acc.setdefault(k, []).append(v)
        t_train = time.time() - t0
        row = {"epoch": epoch, "kimg": trainer.nimg / 1000, "iterations": trainer.it, "time_train_s": round(t_train, 1)}
        row.update({k: float(np.nanmean(v)) if np.isfinite(v).any() else float("nan") for k, v in acc.items()})
        t1 = time.time()
        if epoch % eval_every == 0 or epoch == epochs:
            row.update(engine.evaluate(generate(trainer, z_eval, y_eval), y_eval.cpu(), seed=0))
        else:
            row.update({m: float("nan") for m in METRICS})
        row["time_eval_s"] = round(time.time() - t1, 1)
        hist.append(row)
        atomic_text(pd.DataFrame(hist).to_csv(index=False), hpath)
        v = row.get(PRIMARY)
        if v is not None and np.isfinite(v) and (best["value"] is None or v < best["value"]):
            best = {"value": float(v), "epoch": epoch}
            atomic_torch_save({"G_ema": trainer.G_ema.state_dict(), "epoch": epoch, "metrics": row}, os.path.join(out, "checkpoints", "best.pt"))
        atomic_torch_save({"trainer": trainer.state_dict(), "epoch": epoch, "best": best, "rng": rng_state(), "history": hist}, last)
        if epoch % sample_every == 0 or epoch == epochs:
            save_grid(generate(trainer, z_grid, y_grid), 8, os.path.join(out, "samples", f"epoch_{epoch:04d}.png"))
        eta = (time.time() - t_run) / (epoch - start + 1) * (epochs - epoch) / 3600
        msg = " ".join(f"{m}={row[m]:.4g}" for m in METRICS if np.isfinite(row.get(m, float('nan'))))
        log(f"[{name}] ep {epoch}/{epochs} | G {row.get('loss_G', float('nan')):.3f} D {row.get('loss_D', float('nan')):.3f} | "
            f"{msg} | train {t_train:.0f}s eval {row['time_eval_s']:.0f}s | ETA {eta:.1f} h")
    plot_history(pd.DataFrame(hist), os.path.join(out, "plots"), f"{name}: ")
    atomic_text(json.dumps({"state": "trained", "epochs_target": epochs}), os.path.join(out, "status.json"))
    return out


def final_evaluate(name, make_trainer, data, engine, out_root, n_final=10000, seed=42, extra=None, log=print):
    out = os.path.join(out_root, name)
    trainer = make_trainer()
    b = torch.load(os.path.join(out, "checkpoints", "best.pt"), map_location="cpu", weights_only=False)
    trainer.G_ema.load_state_dict(b["G_ema"])
    z, y = eval_latents(trainer.z_dim(), n_final, data.class_prob, seed + 999, data.device)
    t0 = time.time()
    m = engine.evaluate(generate(trainer, z, y), y.cpu(), seed=0)
    hist = pd.read_csv(os.path.join(out, "history.csv"))
    res = {"run": name, "best_epoch": int(b["epoch"]), "selection": f"min {PRIMARY} on per-epoch eval samples",
           "n_final_samples": n_final, "epochs_trained": int(hist.epoch.max()),
           "kimg": float(hist.kimg.max()), "train_hours": float(hist.time_train_s.sum() / 3600),
           "params": trainer.params(), "metrics": m, "eval_seconds": round(time.time() - t0, 1)}
    if extra: res.update(extra)
    atomic_text(json.dumps(res, indent=2), os.path.join(out, "metrics.json"))
    nc = int(data.y.max()) + 1
    zg = eval_latents(trainer.z_dim(), 8 * nc, data.class_prob, seed + 2, data.device)[0]
    save_grid(generate(trainer, zg, torch.arange(nc, device=data.device).repeat_interleave(8)), 8,
              os.path.join(out, "samples", "final_best.png"))
    atomic_text(json.dumps({"state": "done"}), os.path.join(out, "status.json"))
    log(f"[{name}] final ({n_final} samples, best epoch {b['epoch']}): " + " ".join(f"{k}={v:.4g}" for k, v in m.items()))
    return res

Writing /content/prism_local/code/prism_runner.py


In [6]:
%%writefile /content/prism_local/code/prism_compare.py
"""Comparison tables and plots across any set of runs (baselines now; proposed model and ablations later).

build_comparison({"baseline": ".../baseline_outputs", "proposed": ".../proposed_outputs", "ablation": ...},
                 out_dir, real_reference_json)
reads every <group_dir>/<run>/metrics.json and history.csv that exists and writes to out_dir:
  comparison.csv                 one row per run (+ 'real_data' reference row), all METRICS
  comparison_ranked.csv          rank of each run per metric (1 = best) + mean rank
  bars/<metric>.png              bar chart per metric (font 20, dpi 300)
  curves/<metric>.png            epoch curves of all runs overlaid (font 20, dpi 300)
"""
import os, json, glob
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from prism_metrics import METRICS, DIRECTION, LABELS
from prism_runner import atomic_copy, atomic_text, FONT, DPI


def collect(groups):
    rows, hists = [], {}
    for group, root in groups.items():
        for mj in sorted(glob.glob(os.path.join(root, "*", "metrics.json"))):
            r = json.load(open(mj))
            rows.append({"group": group, "run": r["run"], "best_epoch": r["best_epoch"], "epochs_trained": r["epochs_trained"],
                         "kimg": r["kimg"], "train_hours": r["train_hours"],
                         "params_G_M": r["params"]["G"] / 1e6, **r["metrics"]})
            hp = os.path.join(os.path.dirname(mj), "history.csv")
            if os.path.exists(hp): hists[r["run"]] = pd.read_csv(hp)
    return pd.DataFrame(rows), hists


def _save(fig, path):
    tmp = "/tmp/" + os.path.basename(path)
    fig.savefig(tmp, dpi=DPI, bbox_inches="tight"); plt.close(fig)
    atomic_copy(tmp, path); os.remove(tmp)


def build_comparison(groups, out_dir, real_reference_json=None):
    os.makedirs(os.path.join(out_dir, "bars"), exist_ok=True); os.makedirs(os.path.join(out_dir, "curves"), exist_ok=True)
    df, hists = collect(groups)
    assert len(df), "no metrics.json found"
    ref = json.load(open(real_reference_json)) if real_reference_json and os.path.exists(real_reference_json) else None
    table = df.copy()
    if ref is not None:
        table = pd.concat([table, pd.DataFrame([{"group": "reference", "run": "real_data (test vs train)", **ref}])],
                          ignore_index=True)
    atomic_text(table.to_csv(index=False), os.path.join(out_dir, "comparison.csv"))
    ranks = pd.DataFrame({"group": df.group, "run": df.run})
    for m in METRICS:
        ranks[m] = df[m].rank(ascending=DIRECTION[m] == "min", method="min")
    ranks["mean_rank"] = ranks[METRICS].mean(1)
    atomic_text(ranks.sort_values("mean_rank").to_csv(index=False), os.path.join(out_dir, "comparison_ranked.csv"))

    plt.rcParams.update({"font.size": FONT})
    colors = {"baseline": "#7f7f7f", "proposed": "#d62728", "ablation": "#1f77b4"}
    for m in METRICS:
        fig, ax = plt.subplots(figsize=(max(8, 2.2 * len(df)), 7))
        ax.bar(range(len(df)), df[m], color=[colors.get(g, "#2ca02c") for g in df.group])
        if ref is not None and np.isfinite(ref.get(m, np.nan)):
            ax.axhline(ref[m], ls="--", c="k", lw=2, label="real data (test vs train)"); ax.legend(fontsize=FONT - 4)
        ax.set_xticks(range(len(df))); ax.set_xticklabels(df.run, rotation=30, ha="right")
        arrow = "lower is better" if DIRECTION[m] == "min" else "higher is better"
        ax.set_ylabel(LABELS[m]); ax.set_title(f"{LABELS[m]} ({arrow})"); ax.grid(axis="y", alpha=.3)
        _save(fig, os.path.join(out_dir, "bars", f"{m}.png"))
        fig, ax = plt.subplots(figsize=(11, 7))
        for run, h in hists.items():
            hh = h[["epoch", m]].dropna() if m in h else pd.DataFrame()
            if len(hh): ax.plot(hh.epoch, hh[m], lw=2.5, label=run)
        ax.set_xlabel("Epoch"); ax.set_ylabel(LABELS[m]); ax.set_title(f"{LABELS[m]} ({arrow})")
        ax.grid(alpha=.3); ax.legend(fontsize=FONT - 4)
        _save(fig, os.path.join(out_dir, "curves", f"{m}.png"))
    return table, ranks

Writing /content/prism_local/code/prism_compare.py


In [7]:
import shutil, hashlib, json
def sha256(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()
code_ck = {}
for n in ["prism_metrics.py", "prism_baselines.py", "prism_runner.py", "prism_compare.py"]:
    src, dst = f"{LOCAL}/code/{n}", f"{CODE}/{n}"
    shutil.copyfile(src, dst + ".partial"); os.replace(dst + ".partial", dst)
    code_ck[n] = sha256(src)
json.dump(code_ck, open(f"{CODE}/code_checksums.json", "w"), indent=2)
sys.path.insert(0, f"{LOCAL}/code")
import importlib, prism_metrics, prism_baselines, prism_runner, prism_compare
for m in [prism_metrics, prism_baselines, prism_runner, prism_compare]: importlib.reload(m)
from prism_metrics import MetricEngine, METRICS
from prism_baselines import make_trainer
from prism_runner import TrainData, run, final_evaluate
from prism_compare import build_comparison
print("modules ready:", list(code_ck))

modules ready: ['prism_metrics.py', 'prism_baselines.py', 'prism_runner.py', 'prism_compare.py']


## 3. Data (verified) → GPU

In [8]:
import zipfile, numpy as np, pandas as pd
from PIL import Image
ck_p = json.load(open(f"{DATA}/patch_checksums.json"))
ck_s = json.load(open(f"{DATA}/split_v2_checksums.json"))
assert sha256(f"{DATA}/patches_128.zip") == ck_p["patches_128.zip"]["sha256"], "patches_128.zip checksum mismatch"
assert sha256(f"{DATA}/manifest_v2.csv") == ck_s["manifest_v2.csv"]["sha256"], "manifest_v2.csv checksum mismatch"
MANIFEST_SHA = ck_s["manifest_v2.csv"]["sha256"]
if not os.path.exists(f"{LOCAL}/patches/.done"):
    with zipfile.ZipFile(f"{DATA}/patches_128.zip") as z: z.extractall(f"{LOCAL}/patches")
    open(f"{LOCAL}/patches/.done", "w").close()

man = pd.read_csv(f"{DATA}/manifest_v2.csv", dtype={"item_id": str})
K = man[man.keep == True].copy()
K["class_idx"] = K.class_idx.astype(int)
CLASSES = K.sort_values("class_idx").drop_duplicates("class_idx").pattern.tolist()
def load(df):
    x = np.stack([np.asarray(Image.open(f"{LOCAL}/patches/all/{f}").convert("RGB")) for f in df.file])
    return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous(), torch.tensor(df.class_idx.values)
x_tr, y_tr = load(K[K.split == "train"]); x_te, y_te = load(K[K.split == "test"])
summ = json.load(open(f"{DATA}/split_summary_v2.json"))
assert len(x_tr) == summ["per_split_windows"]["train"] and len(x_te) == summ["per_split_windows"]["test"]
print("classes:", CLASSES, "| train", tuple(x_tr.shape), "| test", tuple(x_te.shape))
print("train per class:", torch.bincount(y_tr).tolist(), "| test per class:", torch.bincount(y_te).tolist())
data = TrainData(x_tr, y_tr, DEVICE, CFG["balance"])
print("class prior used for real/fake labels:", [round(float(p), 3) for p in data.class_prob])

classes: ['floral', 'graphic', 'lattice', 'striped'] | train (3781, 3, 128, 128) | test (953, 3, 128, 128)
train per class: [664, 1896, 295, 926] | test per class: [193, 435, 87, 238]
class prior used for real/fake labels: [0.22, 0.372, 0.147, 0.26]


## 4. Metric engine + real-data cache and reference metrics (computed once, reused by every later notebook)

In [9]:
from prism_runner import atomic_torch_save, atomic_text
engine = MetricEngine(DEVICE, batch_size=100)
cache_path, ref_path = f"{CACHE}/real_cache_v2.pt", f"{CACHE}/real_reference.json"
meta = {"manifest_v2_sha256": MANIFEST_SHA, "n_train": len(x_tr), "n_test": len(x_te), "classes": CLASSES}
use_cache = os.path.exists(cache_path) and os.path.exists(ref_path)
if use_cache:
    engine.load_real_cache(cache_path)
    use_cache = engine.real["meta"] == meta
if not use_cache:
    engine.build_real_cache(x_tr, y_tr, x_te, y_te, meta=meta)
    atomic_torch_save(engine.real, cache_path)
    ref = engine.real_reference(x_te)
    atomic_text(json.dumps(ref, indent=2), ref_path)
ref = json.load(open(ref_path))
print("memorization threshold tau (DINOv2 cosine distance):", round(engine.real["mem_tau"], 4))
print("real data reference (test vs train):"); print(pd.Series(ref).round(4).to_string())

Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:00<00:00, 292MB/s]
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:02<00:00, 235MB/s]
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth
100%|##########| 330M/330M [00:01<00:00, 333MB/s]
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-vgg16-lpips.pth" to /root/.cache/torch/hub/checkpoints/weights-vgg16-lpips.pth
100%|██████████| 7.12k/7.12k [00:00<00:00, 15.8MB/s]


model.safetensors: reconstructing file:   0%|          |  0.00B /  224MB            

model.safetensors: downloading bytes:           |  0.00B            

Downloading: "https://huggingface.co/chaofengc/IQA-PyTorch-Weights/resolve/main/NIMA_InceptionV2_ava-b0c77c00.pth" to /root/.cache/torch/hub/pyiqa/NIMA_InceptionV2_ava-b0c77c00.pth



100%|██████████| 208M/208M [00:02<00:00, 97.4MB/s]


Loading pretrained model NIMA from /root/.cache/torch/hub/pyiqa/NIMA_InceptionV2_ava-b0c77c00.pth
memorization threshold tau (DINOv2 cosine distance): 0.1062
real data reference (test vs train):
fid            52.9691
kid_x1e3        1.6930
fd_dinov2     199.6220
precision       0.5226
recall          0.5393
vendi          37.1306
lpips_div       0.7022
mem_rate        5.0367
nima            3.6871
seam_ratio      3.3729
cond_acc        0.8552


## 5. Train all baselines (resumes automatically)
If the runtime disconnects, rerun cells 0–4 and this cell: every run continues from its last completed epoch.

In [10]:
for name in BASELINES:
    status_p = f"{OUT}/{name}/status.json"
    if os.path.exists(status_p) and json.load(open(status_p))["state"] == "done" and \
       os.path.exists(f"{OUT}/{name}/metrics.json") and json.load(open(f"{OUT}/{name}/metrics.json"))["epochs_trained"] >= EPOCHS:
        print(f"[{name}] done - skipped"); continue
    mk = lambda name=name: make_trainer(name, CFG, DEVICE)
    run(name, mk, CFG, data, engine, OUT, epochs=EPOCHS, eval_every=EVAL_EVERY, n_eval=N_EVAL,
        sample_every=SAMPLE_EVERY, seed=SEED)
    final_evaluate(name, mk, data, engine, OUT, n_final=N_FINAL, seed=SEED,
                   extra={"method": name, "epochs_budget": EPOCHS})

[stylegan2_ada] done - skipped
[diffusion_gan] resumed from epoch 121 (best kid_x1e3=65.80599069595337 @ 120)
[diffusion_gan] ep 122/200 | G 2.093 D 0.651 | fid=144.1 kid_x1e3=65.39 fd_dinov2=2235 precision=0.4575 recall=0.02116 vendi=20.96 lpips_div=0.6843 mem_rate=0 nima=3.212 seam_ratio=2.589 cond_acc=0.4965 | train 125s eval 112s | ETA 5.1 h
[diffusion_gan] ep 123/200 | G 2.111 D 0.652 |  | train 78s eval 0s | ETA 3.4 h
[diffusion_gan] ep 124/200 | G 2.093 D 0.668 | fid=147.7 kid_x1e3=67.62 fd_dinov2=2272 precision=0.475 recall=0.02222 vendi=20.28 lpips_div=0.6851 mem_rate=0 nima=3.207 seam_ratio=2.563 cond_acc=0.494 | train 78s eval 42s | ETA 3.1 h
[diffusion_gan] ep 125/200 | G 2.186 D 0.629 |  | train 78s eval 0s | ETA 2.7 h
[diffusion_gan] ep 126/200 | G 2.084 D 0.636 | fid=147.8 kid_x1e3=66.44 fd_dinov2=2300 precision=0.512 recall=0.02486 vendi=19.99 lpips_div=0.6827 mem_rate=0 nima=3.235 seam_ratio=2.59 cond_acc=0.475 | train 78s eval 42s | ETA 2.6 h
[diffusion_gan] ep 127/20

## 6. Baseline comparison (tables + plots)

In [11]:
table, ranks = build_comparison({"baseline": OUT}, f"{OUT}/comparison", ref_path)
pd.set_option("display.width", 250)
print(table[["group", "run", "best_epoch"] + METRICS].round(4).to_string(index=False))
print(); print(ranks.round(2).to_string(index=False))

    group                       run  best_epoch      fid  kid_x1e3  fd_dinov2  precision  recall   vendi  lpips_div  mem_rate   nima  seam_ratio  cond_acc
 baseline            biggan_diffaug       166.0 112.3804   34.2243  2346.3037     0.5356  0.0034 16.0329     0.6544    0.0000 3.3899      3.5104    0.4189
 baseline             diffusion_gan       188.0 133.8109   57.3140  2061.4396     0.4889  0.0143 21.5303     0.6856    0.0000 3.2756      2.3611    0.4860
 baseline             stylegan2_ada       166.0  87.9427   20.7005  1873.8042     0.4341  0.0135 24.5030     0.6869    0.0000 3.5072      2.7510    0.7043
reference real_data (test vs train)         NaN  52.9691    1.6930   199.6220     0.5226  0.5393 37.1306     0.7022    5.0367 3.6871      3.3729    0.8552

   group            run  fid  kid_x1e3  fd_dinov2  precision  recall  vendi  lpips_div  mem_rate  nima  seam_ratio  cond_acc  mean_rank
baseline biggan_diffaug  2.0       2.0        3.0        1.0     3.0    3.0        3.0  

In [12]:
from IPython.display import Image as IPImage, display
for name in BASELINES:
    p = f"{OUT}/{name}/samples/final_best.png"
    if os.path.exists(p): print(name, "(rows:", CLASSES, ")"); display(IPImage(p, width=700))
display(IPImage(f"{OUT}/comparison/bars/kid_x1e3.png", width=700))

Output hidden; open in https://colab.research.google.com to view.

## 7. Checksums of all outputs, then flush Drive

In [13]:
out_ck = {}
for dp, _, fs in os.walk(OUT):
    for f in fs:
        if f.endswith(".partial") or f == "output_checksums.json": continue
        p = os.path.join(dp, f); out_ck[os.path.relpath(p, OUT)] = {"bytes": os.path.getsize(p), "sha256": sha256(p)}
for f in ["real_cache_v2.pt", "real_reference.json"]:
    out_ck[f"../metric_cache/{f}"] = {"bytes": os.path.getsize(f"{CACHE}/{f}"), "sha256": sha256(f"{CACHE}/{f}")}
atomic_text(json.dumps(out_ck, indent=2), f"{OUT}/output_checksums.json")
print(len(out_ck), "files recorded")
drive.flush_and_unmount()
print("Drive flushed. Runtime -> Disconnect and delete runtime, then run Section B.")

159 files recorded
Drive flushed. Runtime -> Disconnect and delete runtime, then run Section B.


# Section B — verify stored baseline outputs (fresh runtime, CPU is fine)

In [1]:
import os, json, hashlib
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/hari/PRISM-GAN/baseline_outputs"
BASELINES = ["stylegan2_ada", "diffusion_gan", "biggan_diffaug"]
def sha256(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()
ref = json.load(open(f"{OUT}/output_checksums.json"))
bad = [k for k, v in ref.items() if not (os.path.exists(os.path.join(OUT, k)) and
       os.path.getsize(os.path.join(OUT, k)) == v["bytes"] and sha256(os.path.join(OUT, k)) == v["sha256"])]
print(f"{len(ref) - len(bad)}/{len(ref)} files PASS"); [print("FAIL", b) for b in bad]
need = ["history.csv", "metrics.json", "config.json", "checkpoints/last.pt", "checkpoints/best.pt"]
for b in BASELINES:
    miss = [n for n in need if not os.path.exists(f"{OUT}/{b}/{n}")]
    st = json.load(open(f"{OUT}/{b}/status.json"))["state"] if os.path.exists(f"{OUT}/{b}/status.json") else "missing"
    print(f"{'PASS' if not miss and st == 'done' else 'FAIL'}  {b}: status={st} missing={miss}")
print("PASS  comparison.csv" if os.path.exists(f"{OUT}/comparison/comparison.csv") else "FAIL  comparison.csv")
print("\nBASELINE OUTPUTS STORED AND VERIFIED" if not bad else "\nSOME FILES FAILED")

Mounted at /content/drive
159/159 files PASS
PASS  stylegan2_ada: status=done missing=[]
PASS  diffusion_gan: status=done missing=[]
PASS  biggan_diffaug: status=done missing=[]
PASS  comparison.csv

BASELINE OUTPUTS STORED AND VERIFIED
